# CURE-Rec — reviewer revision runbook

This notebook is the execution order for the remaining reviewer-required evidence. It never fabricates values: every expensive action is disabled by default and writes a dedicated run directory.


In [1]:
from pathlib import Path
import importlib
import sys

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None: raise RuntimeError('Open from the CURE-Rec code directory or repository root.')
sys.path.insert(0, str(ROOT))
from cure_rec.config import load_settings
from cure_rec.revision import run_selector_holdout_study
RUN_ROOT = ROOT / 'runs'
FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
print('CURE-Rec source:', ROOT)


CURE-Rec source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## Guarded run controls

Enable one action at a time. The full selector benchmark is already complete; keep it disabled unless changing its protocol.


In [3]:
RUN_HELDOUT_SELECTOR_REPLICATION = True
RUN_WEIGHT_SENSITIVITY = False
RUN_THRESHOLD_SENSITIVITY = False
RUN_CRN_ABLATION = False
RUN_SCALING_STUDY = False
RUN_EXTERNAL_BOOTSTRAP = False
RUN_SECOND_DATASET = False

assert sum((RUN_HELDOUT_SELECTOR_REPLICATION, RUN_WEIGHT_SENSITIVITY, RUN_THRESHOLD_SENSITIVITY, RUN_CRN_ABLATION, RUN_SCALING_STUDY, RUN_EXTERNAL_BOOTSTRAP, RUN_SECOND_DATASET)) <= 1


## Action 1 — independent selector replication

Use only if you intentionally want a second held-out selector benchmark. The completed study already uses selection seeds 42–46 and evaluation seeds 200–219.


In [4]:
if RUN_HELDOUT_SELECTOR_REPLICATION:
    cfg = load_settings(FULL_CONFIG)
    cfg.run.output_root = RUN_ROOT
    run_dir = run_selector_holdout_study(cfg, selection_seeds=(47,48,49,50,51), evaluation_seeds=tuple(range(220,240)))
    print('Independent selector replication:', run_dir)
else:
    print('Selector replication disabled.')


2026-08-09 01:49:41,137 | INFO | run_started | {"config_hash": "5c84efbf88ddbf25", "run_id": "selector-selection-47-20260809T004941Z-9d05b886"}
2026-08-09 01:49:41,137 | INFO | exact_game_started | {}
2026-08-09 01:49:41,138 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-09 01:52:29,655 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13615947266600165, "scenario": "nominal", "shapley_efficiency_gap": 2.7755575615628914e-17}
2026-08-09 01:52:29,656 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-09 02:01:35,850 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13256717826903242, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-09 02:01:35,851 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-09 02:04:20,285 | INFO | scenario_game_completed

## Action 2 — utility-weight sensitivity

Run one predeclared weight vector at a time. Each call is a new held-out selector study; label and archive every vector. Do not choose a favorable vector after inspecting results.


In [ ]:
WEIGHT_VECTOR = {'satisfaction': 0.70, 'retention': 0.30, 'fatigue': 0.35, 'cost': 1.00}
if RUN_WEIGHT_SENSITIVITY:
    cfg = load_settings(FULL_CONFIG)
    cfg.utility.satisfaction_weight = WEIGHT_VECTOR['satisfaction']
    cfg.utility.retention_weight = WEIGHT_VECTOR['retention']
    cfg.utility.fatigue_weight = WEIGHT_VECTOR['fatigue']
    cfg.utility.cost_weight = WEIGHT_VECTOR['cost']
    cfg.run.output_root = RUN_ROOT
    run_dir = run_selector_holdout_study(cfg, selection_seeds=(42,43,44), evaluation_seeds=tuple(range(200,210)))
    print('Weight-sensitivity selector run:', run_dir)
else:
    print('Weight sensitivity disabled.')


## Action 3 — constraint-threshold sensitivity

This tests whether repair decisions are an artifact of one provider/fatigue threshold. Run a predeclared grid externally, one configuration at a time, and retain all outputs.


In [ ]:
THRESHOLDS = {'provider': 0.28, 'fatigue': 0.65, 'relevance': -0.08, 'budget': 0.35}
if RUN_THRESHOLD_SENSITIVITY:
    cfg = load_settings(FULL_CONFIG)
    cfg.constraints.max_provider_disparity = THRESHOLDS['provider']
    cfg.constraints.max_fatigue = THRESHOLDS['fatigue']
    cfg.constraints.min_relevance_delta = THRESHOLDS['relevance']
    cfg.constraints.budget = THRESHOLDS['budget']
    cfg.run.output_root = RUN_ROOT
    run_dir = run_selector_holdout_study(cfg, selection_seeds=(42,43,44), evaluation_seeds=tuple(range(200,210)))
    print('Threshold-sensitivity selector run:', run_dir)
else:
    print('Threshold sensitivity disabled.')


## Actions 4–7 — implementation-gated revision experiments

CRN removal, player-library scaling, user-level bootstrap, and a second-dataset protocol require dedicated implementations beyond the currently validated artifact. They are deliberately not faked or silently approximated by this notebook. Do not enable them until their code has been added and tested.


In [ ]:
if any((RUN_CRN_ABLATION, RUN_SCALING_STUDY, RUN_EXTERNAL_BOOTSTRAP, RUN_SECOND_DATASET)):
    raise NotImplementedError('This reviewer experiment needs its dedicated implementation; do not substitute an unvalidated shortcut.')
print('CRN/scaling/external-bootstrap/second-dataset actions await dedicated code.')


## Archive rule

After every completed reviewer run: inspect outputs, update the manuscript table/figure from the generated files, run Action 7 in the next-actions notebook, verify checksums, then commit and push.
